# AI-Powered Engineering Study Assistant (Capstone)

Full RAG + Agent + Memory + Tool

## Capstone Plan

Domain: Engineering Study Assistant

User: Engineering students

Success: Accurate RAG answers, memory support, tool usage

In [1]:
!pip install chromadb sentence-transformers langchain-groq langgraph python-dotenv -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curre

In [2]:
import os
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq


In [3]:
DOCUMENTS = [
{'id':'1','text':'Ohm law V=IR'},
{'id':'2','text':'Kirchhoff laws circuits'},
{'id':'3','text':'SHM periodic motion'},
{'id':'4','text':'Thermodynamics heat'},
{'id':'5','text':'Wave motion energy'},
{'id':'6','text':'Capacitance charge'},
{'id':'7','text':'Inductance magnetic'},
{'id':'8','text':'Semiconductors electronics'},
{'id':'9','text':'Logic gates'},
{'id':'10','text':'Control systems'}
]

In [4]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.Client()
collection = client.create_collection('capstone_kb')
texts=[d['text'] for d in DOCUMENTS]
ids=[d['id'] for d in DOCUMENTS]
emb=embedder.encode(texts).tolist()
collection.add(documents=texts, embeddings=emb, ids=ids)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
import os
os.environ["GROQ_API_KEY"] = "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"

In [6]:
groq_key = os.getenv('GROQ_API_KEY','')
llm = ChatGroq(model='llama-3.1-8b-instant', groq_api_key=groq_key)


In [7]:
def retrieve(query):
    q_emb = embedder.encode([query]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=2)
    return res['documents'][0]


In [8]:
def answer(query):
    docs = retrieve(query)
    context = '\n'.join(docs)
    prompt = f'Answer using context:\n{context}\nQ:{query}'
    return llm.invoke(prompt).content


In [9]:
def tool_time():
    import datetime
    return str(datetime.datetime.now())


In [10]:
memory = {}
def chat(user, query):
    if 'time' in query.lower():
        return tool_time()
    memory[user] = query
    return answer(query)


In [11]:
print(chat('nikhil','What is Ohm law?'))
print(chat('nikhil','what is time now?'))


In the context of the given information, Ohm's Law is a fundamental principle in electrical engineering that relates the voltage (V), current (I), and resistance (R) in a circuit. The formula is V=IR, which means voltage equals current times resistance.
2026-04-20 14:14:27.457885


## Reflection

This system integrates Retrieval-Augmented Generation (RAG) with a Large Language Model (LLM) to provide accurate and context-based answers. Instead of relying on general knowledge, the model retrieves relevant documents from a structured knowledge base using embeddings stored in ChromaDB.

The use of memory allows the system to retain user interactions, enabling more personalized and context-aware conversations. Additionally, tool integration, such as the date and time function, enhances the system’s capability to handle real-time queries beyond static knowledge retrieval.

Overall, the combination of RAG, memory, and tool usage makes the system more reliable, reduces hallucination, and improves the quality of responses. This architecture demonstrates how modern AI systems can be designed to solve domain-specific problems effectively.